# pandas: loading, cleaning, grouping

**M0 · Toolbox & Submission Craft** — [AI Olympiad Curriculum](https://ai-olympiad-curriculum.netlify.app)

~35 min

## First look at a dataset

A DataFrame is a table with named, typed columns — think of a sheet where every column has a name and a dtype. These calls fit on one screen and should run before anything else:



In [ ]:
import pandas as pd
df = pd.read_csv("train.csv")
df.shape                       # (rows, columns)
df.dtypes                      # dtype per column — watch for "object" that should be numbers
df.head()                      # first 5 rows
df.describe()                  # min/mean/max/quantiles of numeric columns
df.isna().mean()               # share of missing values per column
df["target"].value_counts()    # class balance — is one class 90% of the data?

Two more for the toolbox: `df.info()` (memory use and non-null counts per column) and `df.sample(5)` (random rows — `head()` can hide weirdness that sorted to the bottom of the file).



## Selecting rows and columns

`df[col]` picks one column, `df[[c1, c2]]` several. `df[mask]` filters rows by a boolean Series. `.loc[rows, cols]` does both by **label** (column names are labels); `.iloc[i, j]` does both by **position**.



In [ ]:
df[df.age > 18]                                  # rows where age > 18
df[(df.age > 18) & (df.city == "Almaty")]        # & not and; each condition in ()
df.loc[df.city == "Almaty", ["age", "income"]]   # those rows, only those columns
df.iloc[:5, :3]                                  # first 5 rows, first 3 columns, by position
df[df.city.isin(["A", "B"])]                     # membership test
df.nlargest(10, "income")                        # top-10 rows by a column

- A filtered result is a copy: `df[df.x > 0]["y"] = 1` may warn (SettingWithCopy) and silently write nothing back. Use `df.loc[df.x > 0, "y"] = 1`.
- `df["a"]` on a missing column raises KeyError; `df.a` fails silently when `a` collides with a method name — brackets are safer.



## Cleaning: NaNs, dtypes, duplicates

Real contest data is dirty. `isna()`/`notna()` find holes, `fillna` patches them, `dropna` deletes rows (`axis=1` deletes columns), `astype` converts dtypes, `drop_duplicates` removes repeats, and the `.str` accessor vectorizes string methods over a whole column.



In [ ]:
df["age"] = df["age"].fillna(df["age"].median())   # median, not mean — robust to outliers
df = df.dropna(subset=["target"])                   # rows missing the label are useless
df["price"] = df["price"].astype("float32")
df = df.drop_duplicates(subset=["user_id"], keep="last")
df["city"] = df["city"].str.strip().str.lower()     # " Almaty " → "almaty" for every row

**Why median for NaNs**: a feature with a fat right tail drags the mean toward its outliers; the median sits at the 50th percentile regardless. Even better in contests: add an `age_missing` indicator column — "was missing" is often itself a signal.

- `isna()` won't catch `"N/A"` or `"?"` strings — scan `.unique()` (or `value_counts`) before trusting the NaN count.
- `dropna()` with no arguments drops any row containing any NaN — on wide tables that can delete most of the data.
- Duplicate rows across the train/validation boundary are **leakage** (information from the answer — or from future data — sneaking into the features) — dedupe **before** splitting.



## Grouping and joining



In [ ]:
df.groupby("city")["income"].agg(["mean", "count"])   # summary table: one row per city
df = df.merge(users, on="user_id", how="left")   # users: a second table keyed by user_id
X = pd.get_dummies(df, columns=["city"])         # one-hot: one 0/1 column per city

Two ways to use a group statistic. `groupby(...).mean()` returns **one row per group** (a summary table). `groupby(...).transform("mean")` returns **one value per original row** — each row gets its own group's mean — so the result lands straight back as a new column:



In [ ]:
df.groupby("user_id")["amount"].mean()               # one row per user
df["user_avg"] = df.groupby("user_id")["amount"].transform("mean")   # same length as df
df.pivot_table(index="city", columns="target", values="income", aggfunc="mean")

> **Note.** groupby + agg/transform is the main engine of feature engineering: per-user averages, per-category counts and per-window statistics become columns with no loops.

- After `merge`, always re-check `len(df)` — a duplicated key on the right side silently multiplies rows.
- `how="inner"` drops unmatched rows; `"left"` keeps them with NaNs. Choose deliberately, then re-check `.isna().mean()`.



### ✏️ Which call shows the share of missing values in each column?

**A.** df.count()
**B.** df.describe()
**C.** df.isna().mean()
**D.** df.dropna()

<details><summary>Answer</summary>

**C.** df.isna().mean()

isna() returns True/False per cell; the mean of booleans is the fraction of True.

</details>

### ✏️ You want every transaction row to get that user's average purchase amount. Which call?

**A.** df.pivot_table(index='user_id', values='amount')
**B.** df.groupby('user_id').agg({'amount': 'mean'})
**C.** df.groupby('user_id')['amount'].transform('mean')
**D.** df.sort_values('amount').groupby('user_id').head(1)

<details><summary>Answer</summary>

**C.** df.groupby('user_id')['amount'].transform('mean')

transform returns a result aligned with the original rows; .mean() returns one row per user.

</details>

### ✏️ df.merge(users, on='uid') doubled the row count. Most likely cause?

**A.** df's uid column has NaNs
**B.** left join is the default
**C.** merge always appends new rows
**D.** users contains duplicate uid values

<details><summary>Answer</summary>

**D.** users contains duplicate uid values

A key matching k rows on the right yields k output rows — dedupe or aggregate the right table first.

</details>

---

## ✅ Self-check tasks

Implement the function in the 'your code' cell, then run the check cell — it runs randomized tests and shows the first mismatch. Run the helper cell below once first.

In [ ]:
# ── task-checking helper cell — run once ──
import copy
import numpy as np
import math, statistics, itertools, collections, functools, heapq, random

try:
    import sys; sys.stdout.reconfigure(encoding="utf-8", errors="replace")
except Exception:
    pass

_L = {"noFn":"❓ Define a function `{e}` in the solution cell and run it","runErr":"❌ Test {i}: {t}: {m}","wrong":"❌ Test {i}: {r}","badRef":"⚠️ Check harness broke (generator/reference): {t}: {m}","inL":"Input","expL":"Expected","gotL":"Got","ok":"✅ Correct! All {n} tests passed."}

def _nb_copy_kw(kw):
    return {k: (v.copy() if isinstance(v, np.ndarray) else copy.deepcopy(v)) for k, v in kw.items()}

def _nb_jsafe(x, depth=0):
    if depth > 6: return "…"
    if isinstance(x, np.ndarray): return _nb_jsafe(x.tolist(), depth + 1)
    if isinstance(x, np.integer): return int(x)
    if isinstance(x, np.floating): return float(x)
    if isinstance(x, np.bool_): return bool(x)
    if isinstance(x, float): return round(x, 6) if math.isfinite(x) else str(x)
    if isinstance(x, (list, tuple)): return [_nb_jsafe(v, depth + 1) for v in x]
    if isinstance(x, dict): return {str(k): _nb_jsafe(v, depth + 1) for k, v in x.items()}
    if isinstance(x, (int, str, bool)) or x is None: return x
    return repr(x)

def _nb_cmp(exp, got, path=""):
    if isinstance(exp, np.ndarray) or isinstance(got, np.ndarray):
        e, g = np.asarray(exp), np.asarray(got)
        if e.shape != g.shape:
            return f"shape {tuple(g.shape)} != expected {tuple(e.shape)}"
        try:
            if np.issubdtype(e.dtype, np.number) or np.issubdtype(g.dtype, np.number):
                ef, gf = e.astype(float), g.astype(float)
                close = np.isclose(gf, ef, atol=1e-6, rtol=1e-5, equal_nan=True)
                if not bool(close.all()):
                    i = tuple(int(v) for v in np.argwhere(~close)[0])
                    return f"{path}first mismatch at index {i}: got {gf[i]:.6g} vs expected {ef[i]:.6g}"
            elif not np.array_equal(g, e):
                return f"{path}values differ"
        except Exception as ex:
            return f"{path}cannot compare: {ex}"
        return None
    if isinstance(exp, (int, float)) and not isinstance(exp, bool) and isinstance(got, (int, float)) and not isinstance(got, bool):
        if not np.isclose(float(exp), float(got), atol=1e-6, rtol=1e-5):
            return f"{path}{got} != expected {exp}"
        return None
    if isinstance(exp, (list, tuple)) and isinstance(got, (list, tuple)):
        if len(exp) != len(got):
            return f"{path}length {len(got)} != expected {len(exp)}"
        for i, (a, b) in enumerate(zip(exp, got)):
            r = _nb_cmp(a, b, path + f"[{i}].")
            if r: return r
        return None
    if isinstance(exp, dict) and isinstance(got, dict):
        if set(exp) != set(got):
            return f"{path}keys differ"
        for k in exp:
            r = _nb_cmp(exp[k], got[k], path + f"[{k!r}].")
            if r: return r
        return None
    if exp != got:
        return f"{path}{got!r} != expected {exp!r}"
    return None

def _check_task(g, ref_src, entry, n=20, seed=0):
    """g — globals() ноутбука; ref_src — gen() + эталонное решение."""
    fn = g.get(entry)
    if not callable(fn):
        print(_L["noFn"].format(e=entry)); return
    ns = {"np": np, "math": math, "statistics": statistics, "itertools": itertools,
          "collections": collections, "functools": functools, "heapq": heapq, "random": random}
    try:
        exec(ref_src, ns)
        gen, ref = ns["gen"], ns[entry]
    except Exception as e:
        print(_L["badRef"].format(t=type(e).__name__, m=e)); return
    rng = np.random.default_rng(seed)
    for i in range(n):
        try:
            kw = gen(rng)
        except Exception as e:
            print(_L["badRef"].format(t=type(e).__name__, m=e)); return
        try:
            got = fn(**_nb_copy_kw(kw))
        except Exception as e:
            print(_L["runErr"].format(i=i + 1, t=type(e).__name__, m=e)
                  + "\n" + _L["inL"] + ": " + repr(_nb_jsafe(kw))); return
        exp = ref(**_nb_copy_kw(kw))
        r = _nb_cmp(exp, got)
        if r:
            print(_L["wrong"].format(i=i + 1, r=r)
                  + "\n" + _L["inL"] + ": " + repr(_nb_jsafe(kw))
                  + "\n" + _L["expL"] + ": " + repr(_nb_jsafe(exp))
                  + "\n" + _L["gotL"] + ": " + repr(_nb_jsafe(got))); return
    print(_L["ok"].format(n=n))

### Task 1 — Clip outliers to quantiles

●○○ · ~8 min

> Outlier clipping is a common first step for numeric columns: extreme values distort scaling and can dominate the loss. `np.quantile(x, q)` returns the q-quantile; `np.clip(x, lo, hi)` bounds values.

Implement `clip_outliers(x, q)`: clip the array to its `q`- and `(1-q)`-quantiles.

```
x = [0, 1, 2, 100], q = 0.25  →  clip to [0.75, 26.5]
```

<details><summary>Hint</summary>

lo, hi = np.quantile(x, q), np.quantile(x, 1 - q); then np.clip(x, lo, hi).

</details>


In [ ]:
import numpy as np

def clip_outliers(x, q):
    x = np.asarray(x, dtype=float)
    # your code
    return x

In [ ]:
# self-check — run after solving the cell above
_check_task(globals(), "def gen(rng):\n    n = int(rng.integers(10, 60))\n    x = rng.normal(0, 1, n)\n    x[rng.integers(0, n)] = 100          # inject an outlier\n    return {\"x\": x, \"q\": float(rng.choice([0.01, 0.05, 0.1, 0.25]))}\n\ndef clip_outliers(x, q):\n    x = np.asarray(x, dtype=float)\n    lo, hi = np.quantile(x, q), np.quantile(x, 1 - q)\n    return np.clip(x, lo, hi)", "clip_outliers", 20)

### Task 2 — Median imputation of NaNs

●○○ · ~10 min

> Real olympiad tables are full of missing values, and `np.nan` poisons every reduction it touches (`np.mean` of a column with one NaN is NaN). Median imputation is the robust default: the median ignores outliers, unlike the mean. `np.nanmedian(X, axis=0)` computes per-column medians skipping NaNs; then you fill the missing cells column by column — e.g. via `np.where(np.isnan(X))` indices.

Implement `impute_median(X)` → a copy of `X` (float 2-D array) where every `NaN` is replaced by the median of its column. Do not modify the input in place. Every column is guaranteed to have at least one non-NaN value.

```
[[1, nan], [3, 2], [5, 4]] → [[1, 3], [3, 2], [5, 4]]   (col medians: 3, 3)
```

<details><summary>Hint</summary>

med = np.nanmedian(X, axis=0); rows, cols = np.where(np.isnan(X)); out[rows, cols] = med[cols].

</details>


In [ ]:
import numpy as np

def impute_median(X):
    X = np.asarray(X, dtype=float)
    out = X.copy()
    # your code
    return out

In [ ]:
# self-check — run after solving the cell above
_check_task(globals(), "def gen(rng):\n    n, d = int(rng.integers(6, 20)), int(rng.integers(2, 6))\n    X = rng.normal(0, 3, (n, d))\n    mask = rng.random((n, d)) < 0.25\n    for j in range(d):\n        if mask[:, j].all():\n            mask[int(rng.integers(0, n)), j] = False   # keep >=1 observed per column\n    X[mask] = np.nan\n    return {\"X\": X}\n\ndef impute_median(X):\n    X = np.asarray(X, dtype=float)\n    out = X.copy()\n    med = np.nanmedian(out, axis=0)\n    rows, cols = np.where(np.isnan(out))\n    out[rows, cols] = med[cols]\n    return out", "impute_median", 20)

---

**Go deeper**
- [pandas: 10 minutes to pandas](https://pandas.pydata.org/docs/user_guide/10min.html)